# CBH 參數優化 — XGBoost 代理模型
流程：讀取 JSON → 訓練 XGBoost → Optuna 搜尋最佳參數

In [ ]:
# ── Cell 1：安裝套件 ──
!pip install xgboost optuna -q

In [ ]:
# ── Cell 2：讀取資料 ──
import os, json
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from google.colab import drive

drive.mount('/drive')

# 🎯 讀取加權解答本
weight_json_path = '/drive/MyDrive/OVO-Bench/question_weights.json'
if os.path.exists(weight_json_path):
    with open(weight_json_path, 'r', encoding='utf-8') as f:
        weight_table = json.load(f)
    print(f'📊 成功載入權重解答本，共計 {len(weight_table)} 題')
else:
    weight_table = {}
    print('⚠️ 找不到權重解答本，使用二元正確率')

# 🎯 讀取所有 JSON 結果
results_dir = '/drive/MyDrive/OVO-Bench/ovo_RF_data_v6'
data = []

print('正在讀取資料庫...')
for filename in os.listdir(results_dir):
    if not filename.endswith('.json'):
        continue
    filepath = os.path.join(results_dir, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        try:
            res = json.load(f)
            params = res.get('parameters', {})
            if not params:
                continue
            detail = res.get('detail', [{}])[0]
            q_type = detail.get('q_type', 'UNKNOWN')
            llm_answer = str(detail.get('llm_answer', ''))
            item_id_str = str(res.get('item_id'))
            is_correct = res.get('correct', 0)

            if item_id_str in weight_table:
                weighted_score = weight_table[item_id_str].get(llm_answer, 0.0)
            else:
                weighted_score = 1.0 if is_correct else 0.0

            row = {'item_id': item_id_str, 'q_type': q_type, 'weighted_score': weighted_score}
            row.update(params)
            data.append(row)
        except Exception as e:
            pass

df = pd.DataFrame(data)
print(f'✅ 資料載入成功，共 {len(df)} 筆')
print(f'   題型分布：{df["q_type"].value_counts().to_dict()}')
print(f'   平均加權分數：{df["weighted_score"].mean():.4f}')

In [ ]:
# ── Cell 3：特徵工程 + 訓練 XGBoost ──

# One-Hot Encoding 題型
df_processed = pd.get_dummies(df, columns=['q_type'], prefix='is')
for col in df_processed.columns:
    if col.startswith('is_'):
        df_processed[col] = df_processed[col].astype(int)

# 特徵欄位
all_parameters = [col for col in df.columns if col not in
                  ['item_id', 'q_type', 'weighted_score', 'is_correct']]
type_features  = [c for c in df_processed.columns if c.startswith('is_')]
features = all_parameters + type_features

X = df_processed[features]
y = df_processed['weighted_score']

print(f'特徵數量：{len(features)}')
print(f'特徵列表：{features}')

# 訓練 XGBoost
proxy_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=5,
    random_state=42,
    verbosity=0
)
proxy_model.fit(X, y)

# 訓練集自評
y_pred_train = proxy_model.predict(X)
train_rmse = np.sqrt(np.mean((y - y_pred_train) ** 2))
print(f'\n✅ XGBoost 訓練完畢')
print(f'   訓練集 RMSE：{train_rmse:.4f}')

# 特徵重要性
importances = proxy_model.feature_importances_
imp_df = pd.DataFrame({'Feature': features, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=False)
print('\n🔍 特徵重要性排行：')
for _, row in imp_df.iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"  {row['Feature']:<30}: {row['Importance']:.4f} {bar}")

In [ ]:
# ── Cell 4：Optuna 搜尋最佳參數 ──
optuna.logging.set_verbosity(optuna.logging.WARNING)

all_q_types = ['action', 'attribute', 'spatial']
best_params_all = {}

print('🚀 開始為所有題型尋找最佳參數...\n')

for target_q_type in all_q_types:

    def objective(trial):
        # Optuna 搜尋空間（Magic 7，MAX_N_SMALL → K_SIGMOID）
        realtime_near_n       = trial.suggest_int('REALTIME_NEAR_N', 1, 8)
        big_window_base_sigma = trial.suggest_float('BIG_WINDOW_BASE_SIGMA', 50.0, 150.0)
        dedup_thresh_d        = trial.suggest_float('DEDUP_THRESH_D', 0.85, 0.99)
        stu_move_thresh       = trial.suggest_float('STU_MOVE_THRESH', 0.005, 0.05)
        k_sigmoid             = trial.suggest_float('K_SIGMOID', 2.0, 10.0)
        semantic_near_n       = trial.suggest_int('SEMANTIC_NEAR_N', 1, 8)
        dedup_thresh_a        = trial.suggest_float('DEDUP_THRESH_A', 0.85, 0.99)

        # 建立預測資料
        trial_data = {col: 0.0 for col in features}

        # 填入 Optuna 猜測的參數
        trial_data['REALTIME_NEAR_N']       = realtime_near_n
        trial_data['BIG_WINDOW_BASE_SIGMA'] = big_window_base_sigma
        trial_data['DEDUP_THRESH_D']        = dedup_thresh_d
        trial_data['STU_MOVE_THRESH']       = stu_move_thresh
        trial_data['K_SIGMOID']             = k_sigmoid
        trial_data['SEMANTIC_NEAR_N']       = semantic_near_n
        trial_data['DEDUP_THRESH_A']        = dedup_thresh_a

        # 填入靜態參數
        trial_data['BIG_WINDOW_K']           = 1.0
        trial_data['BIG_WINDOW_EPS']         = 0.01
        trial_data['BIG_WINDOW_BASE_N']      = 16
        trial_data['MIN_INTERVAL_BIG']       = 0.5
        trial_data['MIN_INTERVAL_SMALL']     = 0.25
        trial_data['MAX_N_SMALL']            = 8
        trial_data['SEMANTIC_NEAR_INTERVAL'] = 0.25
        trial_data['REALTIME_NEAR_INTERVAL'] = 0.25
        trial_data['STU_AREA_THRESH']        = 0.10
        trial_data['CLIP_DEDUP_THRESH']      = 0.95
        trial_data['K']                      = 0.3
        trial_data['ALPHA']                  = 1.0
        trial_data['BETA']                   = 1.0

        # 題型 one-hot
        trial_data['is_action']    = 1 if target_q_type == 'action'    else 0
        trial_data['is_attribute'] = 1 if target_q_type == 'attribute' else 0
        trial_data['is_spatial']   = 1 if target_q_type == 'spatial'   else 0

        X_trial = pd.DataFrame([trial_data])[features]
        return proxy_model.predict(X_trial)[0]

    study = optuna.create_study(direction='maximize')
    print(f'⏳ 優化【{target_q_type}】題型 (500 次模擬)...')
    study.optimize(objective, n_trials=500)

    best_params_all[target_q_type] = {
        'score':  study.best_value,
        'params': study.best_params
    }
    print(f'   ✅ 最高預期得分: {study.best_value:.4f}\n')

# ── 輸出結果 ──
print('=' * 60)
print('🏆 XGBoost 黃金參數總結報告')
print('=' * 60)
for q_type, result in best_params_all.items():
    print(f'\n🎯 {q_type.upper()} (預期得分: {result["score"]:.4f})')
    for k, v in result['params'].items():
        if isinstance(v, float):
            print(f'    {k}: {v:.4f}')
        else:
            print(f'    {k}: {v}')

In [ ]:
# ── Cell 5：輸出 OPTUNA_CONFIGS 格式（直接貼進 Cell 1）──
print('\n# 複製以下內容貼進 Cell 1 的 OPTUNA_CONFIGS：\n')
print('OPTUNA_CONFIGS = {')
for q_type, result in best_params_all.items():
    p = result['params']
    print(f"    '{q_type}': {{")
    print(f"        'REALTIME_NEAR_N':       {p['REALTIME_NEAR_N']},")
    print(f"        'BIG_WINDOW_BASE_SIGMA': {p['BIG_WINDOW_BASE_SIGMA']:.4f},")
    print(f"        'DEDUP_THRESH_D':        {p['DEDUP_THRESH_D']:.4f},")
    print(f"        'STU_MOVE_THRESH':       {p['STU_MOVE_THRESH']:.4f},")
    print(f"        'K_SIGMOID':             {p['K_SIGMOID']:.4f},")
    print(f"        'SEMANTIC_NEAR_N':       {p['SEMANTIC_NEAR_N']},")
    print(f"        'DEDUP_THRESH_A':        {p['DEDUP_THRESH_A']:.4f},")
    print(f"    }},")
print('}')